In [1]:
import scanpy as sc
import pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix
import numpy as np

In [5]:
data_dir = "/Users/john/Desktop/mouse_parsebio_20260408"
genes_file = f"{data_dir}/all_genes.csv.gz"
metadata_file = f"{data_dir}/cell_metadata.csv.gz"
mtx_file = f"{data_dir}/count_matrix.mtx"
output_file = f"{data_dir}/mmu.h5ad"

genes = pd.read_csv(genes_file)
metadata = pd.read_csv(metadata_file)
mat = csr_matrix(mmread(mtx_file), dtype=np.float32)

print(f"Matrix: {mat.shape}")
print(f"Genes: {len(genes)}")
print(f"Cells: {len(metadata)}")

Matrix: (109893, 57010)
Genes: 57010
Cells: 109893


In [7]:
# Orient to cells x genes
if mat.shape[0] == len(genes) and mat.shape[1] == len(metadata):
    mat = mat.T.tocsr()

adata = sc.AnnData(X=mat)
adata.obs_names = metadata["bc_wells"].values
adata.var_names = genes["gene_id"].values

# Add all cell metadata
for col in metadata.columns:
    if col != "bc_wells":
        adata.obs[col] = metadata[col].values

# Add all gene metadata
for col in genes.columns:
    adata.var[col] = genes[col].values

print(adata)

AnnData object with n_obs × n_vars = 109893 × 57010
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well'
    var: 'gene_id', 'gene_name', 'genome'


In [8]:
adata.write_h5ad(output_file)
print(f"Saved: {output_file}")

Saved: /Users/suresh/Desktop/mouse_parsebio_20260408/mmu.h5ad
